In [1]:
# Generic imports
import pickle
import numpy as np
import scipy.sparse as sparse
from collections import Counter
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
import seaborn as sns
import colorcet as cc
%matplotlib inline

In [2]:
# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "legend.title_fontsize":16,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.fancybox": True,
    "legend.edgecolor": 'gray',
    "legend.facecolor": 'white',
    "lines.linewidth": 2,
    "lines.markersize": 1,
    #'text.usetex': True
})
figsize = (12,4)
markersize=6
#seaborn_colors = sns.color_palette("muted", n_colors=50)
#seaborn_colors = sns.husl_palette(50, s=.7, l=.6) 
#colors = ListedColormap(seaborn_colors)
glasbey_colors = cc.glasbey[:50]
# Create ListedColormap
colors = ListedColormap(glasbey_colors)


In [3]:
from matplotlib.ticker import FuncFormatter

# Formatter function for 'K' suffix
def thousands_formatter(x, pos):
    return f'{int(x/1000)}K'

In [32]:
save_fig_path = Path("/export/usuarios_ml4ds/lbartolome/NextProcurement/NP-Text_Object/static/Images/paper_procurement")

In [52]:
df = pd.read_parquet("/export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/red_data_outsiders_2024_conTitleCPV_chunks")

In [53]:
# rename ContractFolderStatus.ProcurementProject.RequiredCommodityClassification.ItemClassificationCode to "cpv"
df.rename(columns={'ContractFolderStatus.ProcurementProject.RequiredCommodityClassification.ItemClassificationCode': 'cpv'}, inplace=True)

In [54]:
import numpy as np
import ast

def safe_parse_possible_array_string(item):
    """Fix format like array(['[50300000, 50330000]']) and ignore 'nan' strings."""
    if isinstance(item, np.ndarray) and len(item) == 1:
        string = item[0]
        if isinstance(string, str) and string.strip().lower() == "nan":
            return []  # Treat as empty
        try:
            parsed = ast.literal_eval(string)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            return None
    return None

def extract_cpv_depth(code):
    """Extracts CPV depth from a single CPV code if valid."""
    try:
        code_float = float(code)
        code_str = str(int(code_float))
        return len(code_str.rstrip('0'))
    except (ValueError, TypeError):
        return None

def analyze_cpv_depths(df, column='cpv'):
    depths = []
    format_issues = []
    nan_count = 0
    total = 0

    for item in df[column]:
        
        if isinstance(item, np.ndarray) and len(item) == 1 and str(item[0]).strip().lower() == "nan":
            nan_count += 1
            continue
        
        if isinstance(item, list):
            if len(item) == 0:
                nan_count += 1
                continue
            for code in item:
                total += 1
                depth = extract_cpv_depth(code)
                if depth is not None:
                    depths.append(depth)
                else:
                    format_issues.append(code)

        elif pd.isna(item):
            nan_count += 1

        else:
            # Try to fix malformed numpy array string like: array(['[50300000, 50330000]'])
            recovered_list = safe_parse_possible_array_string(item)
            if recovered_list:
                for code in recovered_list:
                    total += 1
                    depth = extract_cpv_depth(code)
                    if depth is not None:
                        depths.append(depth)
                    else:
                        format_issues.append(code)
            else:
                total += 1
                depth = extract_cpv_depth(item)
                if depth is not None:
                    depths.append(depth)
                else:
                    format_issues.append(item)

    print(f"Total CPV codes processed: {total}")
    print(f"Format issues: {len(format_issues)}")
    print(f"NaNs or empty lists: {nan_count}")
    print(f"Valid CPV codes with depth: {len(depths)}")

    depth_counts = pd.Series(depths).value_counts().sort_index()
    return depth_counts, format_issues, nan_count, total

depth_counts, bad_cpvs, nan_count, total_cpvs = analyze_cpv_depths(df)

print(depth_counts)
print("\nExamples of format issues:")
print(bad_cpvs[:10])  # first 10 malformed entries

def format_latex_count_and_percentage_with_nans(depth_counts, nan_count, total_rows):
    """Formats counts and percent for LaTeX, including a NaN/empty category."""
    full_counts = depth_counts.copy()
    full_counts["NaN / empty"] = nan_count  # Add NaN count as its own category

    formatted = []
    for depth, count in full_counts.items():
        percent = 100 * count / total_rows
        count_str = f"{count:,}".replace(",", r"\,")
        formatted.append((str(depth), f"\\({count_str}\\) ({percent:.2f}\\%)"))

    return pd.DataFrame(formatted, columns=["CPV Code Depth", "Count (Percentage)"])

depth_table = format_latex_count_and_percentage_with_nans(depth_counts, nan_count, total_cpvs)

print(depth_table.to_latex(index=False, escape=False))

# check sum of depth_counts and nan_count
total_count = depth_counts.sum() + nan_count
print(f"Total count of CPV codes (including NaNs): {total_count} (should match total rows in DataFrame: {len(df)})")
assert total_count >= len(df), "Total count of CPV codes (including NaNs) should be greater than or equal to total rows in DataFrame."


/tmp/ipykernel_1367019/3229291093.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  code_float = float(code)


Total CPV codes processed: 46049
Format issues: 0
NaNs or empty lists: 0
Valid CPV codes with depth: 46049
1      568
2     3032
3     6216
4     9461
5    11431
6     9642
7     3953
8     1746
Name: count, dtype: int64

Examples of format issues:
[]
\begin{tabular}{ll}
\toprule
CPV Code Depth & Count (Percentage) \\
\midrule
1 & \(568\) (1.23\%) \\
2 & \(3\,032\) (6.58\%) \\
3 & \(6\,216\) (13.50\%) \\
4 & \(9\,461\) (20.55\%) \\
5 & \(11\,431\) (24.82\%) \\
6 & \(9\,642\) (20.94\%) \\
7 & \(3\,953\) (8.58\%) \\
8 & \(1\,746\) (3.79\%) \\
NaN / empty & \(0\) (0.00\%) \\
\bottomrule
\end{tabular}

Total count of CPV codes (including NaNs): 46049 (should match total rows in DataFrame: 35340)


In [55]:
len(df)

35340

In [27]:
df["len_title"] = df["title"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

In [28]:
df["len_title"].describe()

count    114600.000000
mean         21.944188
std          12.671321
min           1.000000
25%          13.000000
50%          19.000000
75%          28.000000
max         136.000000
Name: len_title, dtype: float64

In [ ]:
nr_extract_error_ppt = len(df[df['texto_tecnico'].str.startswith('[ERROR:')])
nr_extract_error_pcap = len(df[df['texto_administrativo'].str.startswith('[ERROR:')])
nr_ppt_no_possible_download = len(df[df['resultado_tecnico'] != "Descargado correctamente"])
nr_pcap_no_possible_download = len(df[df['resultado_administrativo'] != "Descargado correctamente"])

print(f"Number of PPTs with extraction errors: {nr_extract_error_ppt}")
print(f"Number of PCAPs with extraction errors: {nr_extract_error_pcap}")
print(f"Number of PPTs with no possible download: {nr_ppt_no_possible_download}")
print(f"Number of PCAPs with no possible download: {nr_pcap_no_possible_download}")

Number of PPTs with extraction errors: 4681
Number of PCAPs with extraction errors: 2434
Number of PPTs with no possible download: 2951
Number of PCAPs with no possible download: 4363


In [30]:
# Create CPV code depth table
depth_df = pd.DataFrame(depth_counts.items(), columns=["CPV Code Depth", "Count"])
print(depth_df.to_latex(index=False, caption="CPV Code Depth Distribution", label="tab:cpv_depths"))

# Create title length summary table
title_stats_df = pd.DataFrame({
    "Metric": ["Mean Title Length", "Standard Deviation"],
    "Value": [round(df["len_title"].mean(), 2), round(df["len_title"].std(), 2)]
})
print(title_stats_df.to_latex(index=False, caption="Title Length Summary", label="tab:title_length"))

# Create extraction/download error table
error_df = pd.DataFrame({
    "Issue": [
        "PPTs with extraction errors",
        "PCAPs with extraction errors",
        "PPTs with no possible download",
        "PCAPs with no possible download"
    ],
    "Count": [nr_extract_error_ppt, nr_extract_error_pcap, nr_ppt_no_possible_download, nr_pcap_no_possible_download]
})
print(error_df.to_latex(index=False, caption="Extraction and Download Issues", label="tab:errors"))


\begin{table}
\caption{CPV Code Depth Distribution}
\label{tab:cpv_depths}
\begin{tabular}{rr}
\toprule
CPV Code Depth & Count \\
\midrule
1 & 2706 \\
2 & 6368 \\
3 & 14681 \\
4 & 16248 \\
5 & 15785 \\
6 & 12223 \\
7 & 5219 \\
8 & 3067 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}
\caption{Title Length Summary}
\label{tab:title_length}
\begin{tabular}{lr}
\toprule
Metric & Value \\
\midrule
Mean Title Length & 21.940000 \\
Standard Deviation & 12.670000 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}
\caption{Extraction and Download Issues}
\label{tab:errors}
\begin{tabular}{lr}
\toprule
Issue & Count \\
\midrule
PPTs with extraction errors & 4681 \\
PCAPs with extraction errors & 2434 \\
PPTs with no possible download & 2951 \\
PCAPs with no possible download & 4363 \\
\bottomrule
\end{tabular}
\end{table}



In [32]:
error_df["Percentage"] = (error_df["Count"] / len(df)) * 100
error_df

,Issue,Count,Percentage
0,PPTs with extraction errors,4681,4.084642
1,PCAPs with extraction errors,2434,2.123909
2,PPTs with no possible download,2951,2.575044
3,PCAPs with no possible download,4363,3.807155


In [ ]:
df["len_title"] = df["title"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

plt.figure(figsize=(5, 3), dpi=300)  # Larger figure size
plt.hist(df["len_title"], bins=100, color=colors(3), edgecolor='black')
plt.xlabel("Title Length (words)", fontsize=14)
ax = plt.gca()
ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_edgecolor('black')

plt.ylabel("Frequency", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f"{save_fig_path}/outsiders_title_distribution.png")
plt.show()

In [ ]:
df = pd.read_parquet("/export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/red_data_insiders_2024_conTitle_chunks")

df["len_title"] = df["title"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

plt.figure(figsize=(5, 3), dpi=300)  # Larger figure size
plt.hist(df["len_title"], bins=100, color=colors(3), edgecolor='black')
plt.xlabel("Title Length (words)", fontsize=14)
ax = plt.gca()
ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_edgecolor('black')

plt.ylabel("Frequency", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f"{save_fig_path}/insiders_title_distribution.png")
plt.show()
